## 04 — Fine-Tuning de Stance Detection sobre TweetEval

Fine-tuning de modelos Transformer sobre **TweetEval stance** (cardiffnlp/tweet_eval).
Datos cargados desde `gs://tfm-twitter-processed/` (normalizados por `01_preprocessing.ipynb`).

**Clases:** none/neither (0), against (1), favor (2)  
**Métrica principal:** F1-Average = (F1_favor + F1_against) / 2 — excluye clase *none* (estándar SemEval-2016 Task 6, Sección 3.2.3 TFM)  
**Métrica secundaria:** F1-Macro (comparación con Tabla 2.4 TFM, baseline 69.3%)  
**Tópicos:** abortion · atheism · climate · feminist · hillary (×5)

| # | Estrategia | Modelo HuggingFace | Descripción |
|---|---|---|---|
| S0 | Zero-shot | `cardiffnlp/twitter-roberta-base` | Baseline Tabla 2.4 — sin fine-tuning |
| S1 | Fine-tune ×5 tópicos | `cardiffnlp/twitter-roberta-base` | Mismo backbone, fine-tune por tópico |
| S2 | Fine-tune ×5 tópicos | `vinai/bertweet-base` | 850M tweets EN, fine-tune por tópico |
| S3 | Zero-shot | `cardiffnlp/twitter-xlm-roberta-base-sentiment` | Multilingüe — experimento cross-lingual |

**Nota sobre val sets:** los conjuntos de validación de TweetEval stance son muy pequeños (~40-70 muestras/tópico).
Se mezclan train+val para el fine-tuning y se evalúa únicamente en test.

In [ ]:
# ── Colab setup — ejecutar solo en Google Colab ──────────────────────────
import sys, os
IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IN_COLAB:
    import subprocess
    subprocess.run(['pip', 'install', '-q',
        'transformers', 'datasets', 'evaluate', 'torch',
        'google-cloud-storage', 'accelerate',
        'scikit-learn', 'pyarrow'], check=True)

    import pathlib
    pd_path = pathlib.Path('/content/src/pipeline')
    pd_path.mkdir(parents=True, exist_ok=True)
    (pathlib.Path('/content/src') / '__init__.py').write_text('')
    (pd_path / '__init__.py').write_text('')

    SA = '/content/service-account.json'
    if not os.path.exists(SA):
        raise FileNotFoundError('Sube service-account.json a /content/ y vuelve a ejecutar.')
    print(f'SA encontrada: {SA}')
    print('Setup completo.')
else:
    print('Entorno local — omitido.')

### 1. Instalación de librerías e imports

In [ ]:
%%capture
%pip install transformers datasets evaluate accelerate scikit-learn matplotlib seaborn pyarrow

In [ ]:
import os
import io
import re
import sys
import shutil
import numpy as np
import pandas as pd
from collections import Counter, defaultdict
from pathlib import Path

import torch
import torch.nn as nn

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
import evaluate
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')

# Colab: deshabilitar check torchvision
sys.modules.pop('torchvision', None)
sys.modules.pop('torchvision.io', None)
try:
    import datasets.config as _dc
    _dc.TORCHVISION_AVAILABLE = False
except Exception:
    pass

### 2. Carga del dataset desde GCS

TweetEval stance tiene **5 tópicos** con sus propios splits train/val/test.
Los splits de validación son muy pequeños (~40-70 muestras/tópico) — se mezclan con train.

**Importante:** `#semst` ya fue eliminado en `01_preprocessing.ipynb`.
Verificamos que no aparezca en los datos cargados.

In [ ]:
IN_COLAB = 'google.colab' in sys.modules or Path('/content').exists()

if IN_COLAB:
    from google.cloud import storage
    from google.oauth2 import service_account

    _SA_PATH = '/content/service-account.json'
    assert os.path.exists(_SA_PATH), f'No se encuentra {_SA_PATH}.'
    _creds  = service_account.Credentials.from_service_account_file(_SA_PATH)
    _client = storage.Client(credentials=_creds)

    def _load_parquet(gcs_path, bucket='tfm-twitter-processed'):
        blob = _client.bucket(bucket).blob(gcs_path)
        return pd.read_parquet(io.BytesIO(blob.download_as_bytes()))

    print(f'GCS auth OK')
else:
    _root = Path().resolve()
    for _ in range(6):
        if (_root / 'src').exists() and (_root / 'notebooks').exists():
            break
        _root = _root.parent
    sys.path.insert(0, str(_root / 'src'))
    from pipeline.savers import load_tweeteval_processed as _ltp

    def _load_parquet(gcs_path, bucket='tfm-twitter-processed'):
        task_split = gcs_path.replace('tweeteval/', '').replace('.parquet', '')
        task, split = task_split.rsplit('/', 1)
        return _ltp(task, split, bucket=bucket)

    print(f'Local mode, root: {_root}')

In [ ]:
TOPICS = ['abortion', 'atheism', 'climate', 'feminist', 'hillary']

# Labels: 0=none, 1=against, 2=favor
id2label = {0: 'none', 1: 'against', 2: 'favor'}
label2id = {'none': 0, 'against': 1, 'favor': 2}
label_names = ['none', 'against', 'favor']

# Carga y mezcla train+val por tópico
topic_data = {}  # topic -> {'train_val': Dataset, 'test': Dataset}

print('Cargando splits desde gs://tfm-twitter-processed/tweeteval/stance_*/')
for topic in TOPICS:
    task_key = f'stance_{topic}'
    frames = {}
    for split in ['train', 'validation', 'test']:
        df = _load_parquet(f'tweeteval/{task_key}/{split}.parquet')
        frames[split] = df
        print(f'  {topic:12s} | {split:10s}: {len(df):4d} tweets')

    # Merge train + val (val demasiado pequeño para early stopping fiable)
    df_train_val = pd.concat([frames['train'], frames['validation']], ignore_index=True)
    df_test      = frames['test']

    topic_data[topic] = {
        'train_val': Dataset.from_dict({
            'text':  df_train_val['text'].tolist(),
            'label': df_train_val['label_id'].astype(int).tolist(),
        }),
        'test': Dataset.from_dict({
            'text':  df_test['text'].tolist(),
            'label': df_test['label_id'].astype(int).tolist(),
        }),
        'class_weights': compute_class_weight(
            'balanced',
            classes=np.arange(3),
            y=df_train_val['label_id'].astype(int).values,
        ),
    }

print('\nCarga completa.')

#### 2.1 Verificación: `#semst` eliminado

`#semst` es el hashtag de colección del SemEval-2016 Task 6 — todos los tweets lo tenían.
Si el modelo lo viera, lo memorizaría como señal trivial en lugar de aprender contenido real.
Debe haber sido eliminado por `01_preprocessing.ipynb`.

In [ ]:
semst_found = 0
for topic, data in topic_data.items():
    for split_name in ['train_val', 'test']:
        for text in data[split_name]['text']:
            if '#semst' in text.lower():
                semst_found += 1

if semst_found == 0:
    print('OK — #semst no encontrado en ningún split. Preprocesamiento correcto.')
else:
    print(f'AVISO: #semst encontrado en {semst_found} tweets — revisar 01_preprocessing.ipynb')

### 3. Análisis exploratorio

In [ ]:
fig, axes = plt.subplots(1, len(TOPICS), figsize=(18, 4))
colors = ['#95a5a6', '#e74c3c', '#2ecc71']  # none, against, favor

for ax, topic in zip(axes, TOPICS):
    counts = Counter(topic_data[topic]['train_val']['label'])
    bars = ax.bar(
        [label_names[k] for k in sorted(counts)],
        [counts[k] for k in sorted(counts)],
        color=[colors[k] for k in sorted(counts)],
    )
    ax.set_title(f'{topic}\n(n={len(topic_data[topic]["train_val"])})')
    ax.set_ylabel('Frecuencia')
    for bar, (k, v) in zip(bars, sorted(counts.items())):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                str(v), ha='center', fontsize=9)

plt.suptitle('Distribución de clases — TweetEval Stance (train+val por tópico)', fontsize=13)
plt.tight_layout()
plt.savefig('figures/stance_class_distribution.png', bbox_inches='tight', dpi=120)
plt.show()

In [ ]:
# Tabla resumen: tamaños y distribución por tópico
rows = []
for topic in TOPICS:
    labels_tv = np.array(topic_data[topic]['train_val']['label'])
    labels_te = np.array(topic_data[topic]['test']['label'])
    rows.append({
        'Tópico':    topic,
        'Train+Val': len(labels_tv),
        'Test':      len(labels_te),
        'none (%)':    f"{(labels_tv==0).mean()*100:.1f}",
        'against (%)': f"{(labels_tv==1).mean()*100:.1f}",
        'favor (%)':   f"{(labels_tv==2).mean()*100:.1f}",
    })

df_summary = pd.DataFrame(rows)
print('Resumen por tópico:')
display(df_summary)

### 4. Configuración de modelos

| Modelo | F1-avg (ref. SemEval-2016) | F1-macro (ref. TweetEval p.15) |
|--------|---------------------------|--------------------------------|
| `twitter-roberta-base` (ZS) | — | 68.0% |
| `twitter-roberta-base` (retrained) | — | 69.3% |

Sin grid LR para stance: tarea complementaria, dataset pequeño, un run con LR=2e-5.

In [ ]:
CHECKPOINTS = {
    'S0_twitter-roberta-zeroshot':  'cardiffnlp/twitter-roberta-base',
    'S1_twitter-roberta-finetune':  'cardiffnlp/twitter-roberta-base',
    'S2_bertweet-finetune':         'vinai/bertweet-base',
    'S3_xlm-roberta-zeroshot':      'cardiffnlp/twitter-xlm-roberta-base-sentiment',
}

ZERO_SHOT_KEYS = ['S0_twitter-roberta-zeroshot', 'S3_xlm-roberta-zeroshot']
FINETUNE_KEYS  = ['S1_twitter-roberta-finetune', 'S2_bertweet-finetune']

cfg = {
    'max_length':      128,
    'num_labels':      3,
    'batch_size':      16,     # dataset pequeño — batch menor
    'num_epochs':      5,
    'learning_rate':   2e-5,
    'weight_decay':    0.01,
    'checkpoints_dir': '/tmp/stance_checkpoints',
}

print('Modelos:',       list(CHECKPOINTS.keys()))
print('Zero-shot:',     ZERO_SHOT_KEYS)
print('Fine-tune ×5:', FINETUNE_KEYS)

### 5. Métricas

**F1-Average** (SemEval-2016 Task 6): promedio macro sobre clases *favor* y *against* únicamente.  
La clase *none/neither* se excluye porque su presencia domina artificialmente la métrica.  

**F1-Macro** (TweetEval): promedio sobre las 3 clases — para comparar con Tabla 2.4.

In [ ]:
def f1_average(y_true, y_pred):
    """F1-Average SemEval-2016: (F1_against + F1_favor) / 2, excluye none (0)."""
    f1_per = f1_score(y_true, y_pred, average=None, labels=[0, 1, 2], zero_division=0)
    return (f1_per[1] + f1_per[2]) / 2  # against + favor only

def compute_stance_metrics(y_true, y_pred):
    f1_per  = f1_score(y_true, y_pred, average=None, labels=[0, 1, 2], zero_division=0)
    return {
        'f1_average': (f1_per[1] + f1_per[2]) / 2,  # métrica principal
        'f1_macro':   f1_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_none':    f1_per[0],
        'f1_against': f1_per[1],
        'f1_favor':   f1_per[2],
        'accuracy':   accuracy_score(y_true, y_pred),
    }

# compute_metrics para Trainer (eval_strategy='no' → no se usa durante entrenamiento)
def compute_metrics_trainer(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    m = compute_stance_metrics(labels, preds)
    return {'f1_average': m['f1_average'], 'f1_macro': m['f1_macro']}

print('Métricas definidas: f1_average (principal), f1_macro (comparación Tabla 2.4)')

### 6. WeightedTrainer y función de entrenamiento por tópico

**Sin val set** durante entrenamiento (train+val mezclados → no hay split de evaluación intermedia).  
Se entrena por épocas fijas con `eval_strategy='no'` y se evalúa solo en test al finalizar.

In [ ]:
class WeightedTrainer(Trainer):
    """Trainer con CrossEntropyLoss ponderada para clases desequilibradas."""
    def __init__(self, class_weights, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop('labels')
        outputs = model(**inputs)
        loss_fn = nn.CrossEntropyLoss(
            weight=self.class_weights.to(outputs.logits.device)
        )
        loss = loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

In [ ]:
def run_stance_strategy(strategy_name, model_checkpoint, topic,
                        train_val_ds, test_ds, class_weights_arr,
                        cfg_base, zero_shot=False, seed=SEED):
    """
    Evalúa una estrategia sobre un tópico de stance.
    zero_shot=True: solo predict en test sin fine-tuning.
    zero_shot=False: fine-tune en train_val, predict en test.
    """
    label = f'{strategy_name} | {topic}'
    mode  = 'zero-shot' if zero_shot else 'fine-tune'
    print(f'\n  [{mode}] {label}')

    tok = AutoTokenizer.from_pretrained(model_checkpoint)

    def _tokenize(examples):
        return tok(examples['text'], truncation=True,
                   max_length=cfg_base['max_length'], padding=False)

    def prep(ds):
        t = ds.map(_tokenize, batched=True)
        t = t.rename_column('label', 'labels')
        cols = ['input_ids', 'attention_mask', 'labels']
        if 'token_type_ids' in t.column_names:
            cols.append('token_type_ids')
        t.set_format('torch', columns=cols)
        return t

    tr = prep(train_val_ds)
    te = prep(test_ds)

    m = AutoModelForSequenceClassification.from_pretrained(
        model_checkpoint,
        num_labels=3,
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True,
    ).to(device)

    dc = DataCollatorWithPadding(tokenizer=tok)
    cw = torch.tensor(class_weights_arr, dtype=torch.float)

    output_dir = f'{cfg_base["checkpoints_dir"]}_{strategy_name}_{topic}'

    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=0 if zero_shot else cfg_base['num_epochs'],
        per_device_train_batch_size=cfg_base['batch_size'],
        per_device_eval_batch_size=cfg_base['batch_size'],
        learning_rate=cfg_base['learning_rate'],
        weight_decay=cfg_base['weight_decay'],
        eval_strategy='no',   # sin val set (train+val mezclados)
        save_strategy='no',
        logging_steps=50,
        seed=seed,
        report_to='none',
    )

    trainer = WeightedTrainer(
        class_weights=cw,
        model=m,
        args=training_args,
        train_dataset=tr,
        processing_class=tok,
        data_collator=dc,
        compute_metrics=compute_metrics_trainer,
    )

    if not zero_shot:
        trainer.train()

    preds_out = trainer.predict(te)
    y_p = np.argmax(preds_out.predictions, axis=-1)
    y_t = preds_out.label_ids

    metrics = compute_stance_metrics(y_t, y_p)
    print(f'  F1-avg={metrics["f1_average"]*100:.2f}%  '
          f'F1-macro={metrics["f1_macro"]*100:.2f}%  '
          f'Acc={metrics["accuracy"]*100:.2f}%')

    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)

    return {
        'strategy':  strategy_name,
        'topic':     topic,
        'mode':      mode,
        'trainer':   trainer,
        'tokenizer': tok,
        **metrics,
        'y_pred':    y_p,
        'y_true':    y_t,
        'logits':    preds_out.predictions,
    }

print('run_stance_strategy listo.')

### 7. S0 — Zero-shot: `twitter-roberta-base`

Replica el baseline de **Tabla 2.4** del TFM (F1-macro 68.0%).
Sin fine-tuning: el modelo usa sus pesos preentrenados directamente para clasificar stance.

In [ ]:
results_s0 = {}
print('=== S0 — Zero-shot twitter-roberta-base ===')
for topic in TOPICS:
    res = run_stance_strategy(
        strategy_name   = 'S0_TwitterRoBERTa_zeroshot',
        model_checkpoint= CHECKPOINTS['S0_twitter-roberta-zeroshot'],
        topic           = topic,
        train_val_ds    = topic_data[topic]['train_val'],
        test_ds         = topic_data[topic]['test'],
        class_weights_arr = topic_data[topic]['class_weights'],
        cfg_base        = cfg,
        zero_shot       = True,
    )
    results_s0[topic] = res

f1_avg_mean = np.mean([r['f1_average'] for r in results_s0.values()])
f1_mac_mean = np.mean([r['f1_macro']   for r in results_s0.values()])
print(f'\nS0 media: F1-avg={f1_avg_mean*100:.2f}%  F1-macro={f1_mac_mean*100:.2f}%')

### 8. S3 — Zero-shot: `twitter-xlm-roberta-base-sentiment`

Modelo multilingüe — evalúa cuánto pierde respecto a S0 sin fine-tuning.
Su valor real aparecerá en notebook 05 al inferir sobre España 2023.

In [ ]:
results_s3 = {}
print('=== S3 — Zero-shot twitter-xlm-roberta-base-sentiment ===')
for topic in TOPICS:
    res = run_stance_strategy(
        strategy_name   = 'S3_XLM-RoBERTa_zeroshot',
        model_checkpoint= CHECKPOINTS['S3_xlm-roberta-zeroshot'],
        topic           = topic,
        train_val_ds    = topic_data[topic]['train_val'],
        test_ds         = topic_data[topic]['test'],
        class_weights_arr = topic_data[topic]['class_weights'],
        cfg_base        = cfg,
        zero_shot       = True,
    )
    results_s3[topic] = res

f1_avg_mean = np.mean([r['f1_average'] for r in results_s3.values()])
f1_mac_mean = np.mean([r['f1_macro']   for r in results_s3.values()])
print(f'\nS3 media: F1-avg={f1_avg_mean*100:.2f}%  F1-macro={f1_mac_mean*100:.2f}%')

### 9. S1 — Fine-tune: `twitter-roberta-base` (por tópico)

Fine-tuning separado para cada uno de los 5 tópicos.
Objetivo: superar el baseline S0 y el RoBERTa-Retrained (69.3% F1-macro, Tabla 2.4).

In [ ]:
results_s1 = {}
print('=== S1 — Fine-tune twitter-roberta-base × 5 tópicos ===')
for topic in TOPICS:
    res = run_stance_strategy(
        strategy_name   = 'S1_TwitterRoBERTa_finetune',
        model_checkpoint= CHECKPOINTS['S1_twitter-roberta-finetune'],
        topic           = topic,
        train_val_ds    = topic_data[topic]['train_val'],
        test_ds         = topic_data[topic]['test'],
        class_weights_arr = topic_data[topic]['class_weights'],
        cfg_base        = cfg,
        zero_shot       = False,
    )
    results_s1[topic] = res

f1_avg_mean = np.mean([r['f1_average'] for r in results_s1.values()])
f1_mac_mean = np.mean([r['f1_macro']   for r in results_s1.values()])
print(f'\nS1 media: F1-avg={f1_avg_mean*100:.2f}%  F1-macro={f1_mac_mean*100:.2f}%')

### 10. S2 — Fine-tune: `bertweet-base` (por tópico)

BERTweet preentrenado en 850M tweets EN — más corpus Twitter que S1.
Comparar con S1 mide si el mayor corpus de preentrenamiento ayuda en stance.

In [ ]:
results_s2 = {}
print('=== S2 — Fine-tune bertweet-base × 5 tópicos ===')
for topic in TOPICS:
    res = run_stance_strategy(
        strategy_name   = 'S2_BERTweet_finetune',
        model_checkpoint= CHECKPOINTS['S2_bertweet-finetune'],
        topic           = topic,
        train_val_ds    = topic_data[topic]['train_val'],
        test_ds         = topic_data[topic]['test'],
        class_weights_arr = topic_data[topic]['class_weights'],
        cfg_base        = cfg,
        zero_shot       = False,
    )
    results_s2[topic] = res

f1_avg_mean = np.mean([r['f1_average'] for r in results_s2.values()])
f1_mac_mean = np.mean([r['f1_macro']   for r in results_s2.values()])
print(f'\nS2 media: F1-avg={f1_avg_mean*100:.2f}%  F1-macro={f1_mac_mean*100:.2f}%')

### 11. Tabla de resultados

Comparativa completa: F1-Average (métrica principal SemEval) y F1-Macro (referencia TweetEval Tabla 2.4).

In [ ]:
# Reunir todos los resultados
all_results = {
    'S0': results_s0,
    'S1': results_s1,
    'S2': results_s2,
    'S3': results_s3,
}

rows = []
for skey, topic_results in all_results.items():
    for topic, r in topic_results.items():
        rows.append({
            'Estrategia':   skey,
            'Tópico':       topic,
            'Modo':         r['mode'],
            'F1-avg (%)':   round(r['f1_average'] * 100, 2),
            'F1-macro (%)': round(r['f1_macro']   * 100, 2),
            'F1-against':   round(r['f1_against'] * 100, 2),
            'F1-favor':     round(r['f1_favor']   * 100, 2),
            'F1-none':      round(r['f1_none']    * 100, 2),
            'Accuracy (%)': round(r['accuracy']   * 100, 2),
        })

df_results = pd.DataFrame(rows)
display(df_results)

In [ ]:
# Tabla pivotada: estrategia × tópico (F1-avg)
df_pivot_avg = df_results.pivot_table(
    index='Estrategia', columns='Tópico', values='F1-avg (%)', aggfunc='mean'
).round(2)
df_pivot_avg['Media'] = df_pivot_avg.mean(axis=1).round(2)

print('F1-Average por estrategia y tópico (%):')
print(f'  [Ref. TweetEval RoBERTa-Retrained F1-macro: 69.3%]')
display(df_pivot_avg.style.highlight_max(axis=0, color='#d4efdf'))
print(df_pivot_avg.to_latex(caption='F1-Average por estrategia y tópico — TweetEval Stance'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
x = np.arange(len(TOPICS))
width = 0.2
strategy_colors = {'S0': '#3498db', 'S1': '#e67e22', 'S2': '#2ecc71', 'S3': '#9b59b6'}

for ax, metric in zip(axes, ['F1-avg (%)', 'F1-macro (%)']):
    for i, skey in enumerate(all_results):
        vals = [df_results[(df_results['Estrategia']==skey) &
                           (df_results['Tópico']==t)][metric].values[0] for t in TOPICS]
        ax.bar(x + i*width - 1.5*width, vals, width, label=skey,
               color=strategy_colors[skey], alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(TOPICS, rotation=15)
    ax.set_ylabel(metric)
    ax.set_title(f'{metric} por tópico')
    ax.legend()
    ax.axhline(69.3, color='red', linestyle='--', linewidth=1,
               label='Ref. RoBERTa-Retrained (69.3%)')
    ax.legend(fontsize=8)

plt.suptitle('Comparación estrategias — TweetEval Stance', fontsize=13)
plt.tight_layout()
plt.savefig('figures/stance_strategies_comparison.png', bbox_inches='tight', dpi=120)
plt.show()

In [ ]:
# Matrices de confusión: mejor estrategia FT (S1) por tópico
fig, axes = plt.subplots(1, len(TOPICS), figsize=(20, 4))

for ax, topic in zip(axes, TOPICS):
    r  = results_s1[topic]
    cm = confusion_matrix(r['y_true'], r['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=label_names, yticklabels=label_names, cbar=False)
    ax.set_title(f'{topic}\nF1-avg={r["f1_average"]*100:.1f}%', fontsize=9)
    ax.set_ylabel('Real' if topic == TOPICS[0] else '')
    ax.set_xlabel('Predicho')

plt.suptitle('Matrices de confusión — S1 twitter-roberta-base fine-tune', fontsize=12)
plt.tight_layout()
plt.savefig('figures/stance_confusion_matrices_s1.png', bbox_inches='tight', dpi=120)
plt.show()

### 12. Selección y guardado del mejor modelo por tópico en GCS

El mejor modelo por tópico (S1 o S2 según F1-avg) se guarda en
`gs://tfm-twitter-curated/models/stance/{topic}/` para su uso en `05_stance_scraped.ipynb`.

In [ ]:
best_by_topic = {}

for topic in TOPICS:
    candidates = {
        'S1': results_s1[topic],
        'S2': results_s2[topic],
    }
    best_key = max(candidates, key=lambda k: candidates[k]['f1_average'])
    best_by_topic[topic] = candidates[best_key]
    print(f'{topic:12s} → {best_key}  '
          f'F1-avg={candidates[best_key]["f1_average"]*100:.2f}%')

overall_f1avg = np.mean([r['f1_average'] for r in best_by_topic.values()])
print(f'\nMedia global F1-avg (best per topic): {overall_f1avg*100:.2f}%')

In [ ]:
from google.cloud import storage as gcs_lib
from google.oauth2 import service_account as sa_lib

_SA_PATH = '/content/service-account.json' if IN_COLAB else str(
    next(_root.rglob('service-account.json'), '')
)
_gcreds  = sa_lib.Credentials.from_service_account_file(_SA_PATH)
_gclient = gcs_lib.Client(credentials=_gcreds)

for topic, r in best_by_topic.items():
    local_dir = f'/tmp/stance_{topic}_best'
    r['trainer'].save_model(local_dir)
    r['tokenizer'].save_pretrained(local_dir)

    bucket = _gclient.bucket('tfm-twitter-curated')
    for fpath in Path(local_dir).rglob('*'):
        if fpath.is_file():
            gcs_path = f'models/stance/{topic}/{fpath.relative_to(local_dir)}'
            bucket.blob(gcs_path).upload_from_filename(str(fpath))
            print(f'  ↑ {gcs_path}')

    shutil.rmtree(local_dir)
    print(f'{topic} guardado en gs://tfm-twitter-curated/models/stance/{topic}/')

print('\nTodos los modelos guardados.')

### 13. Análisis de resultados

Comparativa entre tópicos, análisis de errores por clase y visualizaciones para Cap 5.

In [ ]:
# Heatmap: F1 por clase y tópico para S1
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
metric_keys = ['f1_none', 'f1_against', 'f1_favor']
metric_labels = ['F1-none', 'F1-against', 'F1-favor']

for ax, mk, ml in zip(axes, metric_keys, metric_labels):
    data = [[results_s1[t][mk] * 100 for t in TOPICS]]
    sns.heatmap(data, annot=True, fmt='.1f', cmap='YlGn', ax=ax,
                xticklabels=TOPICS, yticklabels=['S1'],
                vmin=0, vmax=100, cbar=False)
    ax.set_title(ml)
    ax.set_xticklabels(TOPICS, rotation=30)

plt.suptitle('F1 por clase — S1 twitter-roberta-base fine-tune (%)', fontsize=12)
plt.tight_layout()
plt.savefig('figures/stance_per_class_f1.png', bbox_inches='tight', dpi=120)
plt.show()

In [ ]:
# Zero-shot vs Fine-tune: ganancia por tópico
print('Ganancia F1-avg fine-tune vs zero-shot (S1 vs S0):')
print(f'  {"Tópico":12s}  {"S0 (ZS)":>10s}  {"S1 (FT)":>10s}  {"Delta":>8s}')
print('-' * 48)
for topic in TOPICS:
    s0 = results_s0[topic]['f1_average'] * 100
    s1 = results_s1[topic]['f1_average'] * 100
    delta = s1 - s0
    sign = '+' if delta >= 0 else ''
    print(f'  {topic:12s}  {s0:>10.2f}  {s1:>10.2f}  {sign}{delta:>7.2f}')

In [ ]:
# Classification reports detallados por tópico (mejor modelo)
for topic in TOPICS:
    r = best_by_topic[topic]
    print(f'\n── {topic.upper()} ({r["strategy"]}) ──')
    print(classification_report(r['y_true'], r['y_pred'],
                                target_names=label_names, zero_division=0))